**Лабораторный практикум по курсу «Распознавание диктора», Университет ИТМО, 2025**		

**Лабораторная работа №6. Обучение блока построения дикторских моделей на основе трансформерных претрейнов**

**Цель работы:** изучение процедуры обучения блока построения дикторских моделей на основе трансформерных претрейнов.

**Краткое описание:** в рамках настоящей лабораторной работы предлагается изучить и реализовать схему обучения нейросетевого блока построения дикторских моделей на основе Whisper tiny трансформерного претрейна. Процедуры обучения и тестирования предлагается выполнить по отношению к задаче идентификации на закрытом множестве и верификации на открытом множестве, соответственно. Тестирование полученной системы предполагает использование доли правильных ответов (accuracy) в рамках идентификации диктора на закрытом множестве, а также равновероятной ошибки (EER, equal error rate) в рамках верификации диктора на открытом множестве в качестве целевых метрик оценки качества.

**Данные:** в качестве данных для выполнения лабораторной работы предлагается использовать базу [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы**

1. Подготовка данных для обучения и тестирования блока построения дикторских моделей.							

2. Обучение параметров блока построения дикторских моделей, основанного на использовании «размороженного» Whisper tiny трансформерного претрейна.

3. Проведение дополнительных экспериментов по использованию трансформерных архитектур в задаче распознавания диктора.

In [2]:
# IPython extension to reload modules before executing user code
%load_ext autoreload
%autoreload 2

# Import of modules
import os
import sys
import itertools
from pathlib import Path
from collections import OrderedDict

sys.path.append(os.path.realpath('..'))

import numpy as np

if not hasattr(np, "float"):
    np.float = float

if not hasattr(np, "int"):
    np.int = int

if not hasattr(np, "complex"):
    np.complex = complex
    
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from common import download_dataset, concatenate, extract_dataset, part_extract, download_protocol, split_musan
from common import test_dataset_loader as test_dataset_loader_verification
from common import extract_features, compute_scores
from common import get_eer
from exercises_blank import train_dataset_loader, test_dataset_loader, train_network, test_network, tar_imp_hists
from exercises_blank import WhisperEncoder, TDNNSimple, StatPoolLayer, MaxoutSegmentLevel, SSLDownstream, MainModel
from whisper_fbanks import ExtractWhisperFbanks80
from LossFunction import AAMSoftmaxLoss
from Optimizer import SGDOptimizer, AdamOptimizer
from Scheduler import OneCycleLRScheduler
from load_save_pth import saveParameters, loadParameters

/home/ext-ivanova-mk@ad.speechpro.com/work_dir/srlab_venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ext-ivanova-mk@ad.speechpro.com/work_dir/srlab_venv/lib/python3.8/site-packages/whisper/timing.py:58: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def backtrace(trace: np.ndarray):


**1. Подготовка данных для обучения и тестирования детектора речевой активности**

В ходе выполнения лабораторной работы необходимы данные для проведения процедуры обучения и процедуры тестирования нейросетевого блока генерации дикторских моделей. Возьмём в качестве этих данных звукозаписи, сохраненные в формат *wav*, из корпуса [VoxCeleb1 dev set и VoxCeleb1 test set](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html). Данные корпуса содержат 148,642 и 4,874 звукозаписи (частота дискретизации равна 16кГц) с общим количеством дикторов женского и мужского пола равным 1,251, разговаривающих преимущественно на английском языке. В рамках настоящей лабораторной работы данные из корпуса VoxCeleb1 dev set  будут использоваться при проведения процедур обучения и валидации блока построения дикторских моделей в задаче идентификации на закрытом множестве. Данные из корпуса VoxCeleb1 test set будут использоваться для проведения верификационного теста на открытом множестве из 40 дикторов, которые отсутствовали в корпусе VoxCeleb1 dev set.

В рамках настоящего пункта требуется выполнить следующее:

1. Загрузить и распаковать звуковые wav-файлы из корпуса VoxCeleb1 dev set и VoxCeleb1 test set.

2. Загрузить идентификационный протокол, позволяющий разделить данные из корпуса VoxCeleb1 dev set на обучающее и валидацилонное множества.

3. Загрузить верификационный протокол, связанный с корпусом VoxCeleb1 test set.

4. Загрузить и распаковать данные из корпусов [SLR17 (MUSAN)](https://openslr.org/17/) and [SLR28 (RIR noises)](https://openslr.org/28/), с использованием которых можно расширить обучающее множество данных через запуск процедур аугментации.

![Рисунок 1](../images/voxceleb_dataset.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [4]:
from pathlib import Path

DATA_ROOT = Path("/mnt/storage/work_dir/databases")

VOX1_DEV_ROOT  = DATA_ROOT / "voxceleb1_dev"
VOX1_TEST_ROOT = DATA_ROOT / "voxceleb1_test"

MUSAN_SPLIT_ROOT = DATA_ROOT / "musan_split"
MUSAN_ROOT       = DATA_ROOT / "musan"

RIR_ROOT = DATA_ROOT / "RIRS_NOISES" / "simulated_rirs"


def find_wav_dir(root: Path) -> Path:
    """
    Находит папку, внутри которой лежат директории дикторов:
    id10001, id10002, ...
    """
    candidates = [
        root / "wav",
        root,
        root / "voxceleb1" / "wav",
        root / "voxceleb1_dev" / "wav",
        root / "voxceleb1_test" / "wav",
    ]

    for cand in candidates:
        if cand.exists() and cand.is_dir():
            has_speakers = any(
                p.is_dir() and p.name.startswith("id")
                for p in cand.iterdir()
            )
            if has_speakers:
                return cand

    raise FileNotFoundError(f"Cannot find VoxCeleb wav directory under {root}")


def find_file(root: Path, *names: str) -> Path:
    """
    Ищет файл по нескольким возможным именам.
    Например: iden_split.txt или iden_split.
    """
    for name in names:
        direct = root / name
        if direct.exists():
            return direct

        matches = list(root.rglob(name))
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Cannot find any of {names} under {root}")


dev_wav_dir = find_wav_dir(VOX1_DEV_ROOT)
test_wav_dir = find_wav_dir(VOX1_TEST_ROOT)

iden_split_path = find_file(
    VOX1_TEST_ROOT,
    "iden_split.txt",
    "iden_split",
)

veri_test2_path = find_file(
    VOX1_TEST_ROOT,
    "veri_test2.txt",
    "veri_test2",
)

# Пути, которые дальше использует ноутбук.
train_path = str(dev_wav_dir)
val_path   = str(dev_wav_dir)
test_path  = str(test_wav_dir) + "/"

# MUSAN: если уже есть musan_split, используем его.
# Если есть только musan, то split_musan надо выполнить один раз отдельно.
if MUSAN_SPLIT_ROOT.exists():
    musan_path = str(MUSAN_SPLIT_ROOT)
elif MUSAN_ROOT.exists():
    musan_path = str(MUSAN_ROOT)
else:
    musan_path = None

# RIR noises.
rir_path = str(RIR_ROOT) if RIR_ROOT.exists() else None

print("DATA_ROOT:", DATA_ROOT)
print("dev wav:", dev_wav_dir)
print("test wav:", test_wav_dir)
print("iden_split:", iden_split_path)
print("veri_test2:", veri_test2_path)
print("musan_path:", musan_path)
print("rir_path:", rir_path)

DATA_ROOT: /mnt/storage/work_dir/databases
dev wav: /mnt/storage/work_dir/databases/voxceleb1_dev/wav
test wav: /mnt/storage/work_dir/databases/voxceleb1_test
iden_split: /mnt/storage/work_dir/databases/voxceleb1_test/iden_split.txt
veri_test2: /mnt/storage/work_dir/databases/voxceleb1_test/veri_test2.txt
musan_path: /mnt/storage/work_dir/databases/musan_split
rir_path: /mnt/storage/work_dir/databases/RIRS_NOISES/simulated_rirs


In [5]:
assert dev_wav_dir.exists(), dev_wav_dir
assert test_wav_dir.exists(), test_wav_dir
assert iden_split_path.exists(), iden_split_path
assert veri_test2_path.exists(), veri_test2_path

dev_speakers = [
    p for p in dev_wav_dir.iterdir()
    if p.is_dir() and p.name.startswith("id")
]

test_speakers = [
    p for p in test_wav_dir.iterdir()
    if p.is_dir() and p.name.startswith("id")
]

print("VoxCeleb1 dev speakers:", len(dev_speakers))
print("VoxCeleb1 test speakers:", len(test_speakers))

assert len(dev_speakers) > 0
assert len(test_speakers) > 0

with open(iden_split_path, "r", encoding="utf-8") as f:
    iden_lines = [line.strip() for line in f if line.strip()]

with open(veri_test2_path, "r", encoding="utf-8") as f:
    veri_lines = [line.strip() for line in f if line.strip()]

print("iden_split lines:", len(iden_lines))
print("veri_test2 lines:", len(veri_lines))

assert len(iden_lines) > 0
assert len(veri_lines) > 0

# Проверяем несколько файлов из iden_split.
for line in iden_lines[:20]:
    split_id, rel_path = line.split()[:2]
    wav_path = dev_wav_dir / rel_path
    assert wav_path.exists(), wav_path

# Проверяем несколько файлов из veri_test2.
for line in veri_lines[:20]:
    parts = line.split()
    wav_1 = test_wav_dir / parts[-2]
    wav_2 = test_wav_dir / parts[-1]
    assert wav_1.exists(), wav_1
    assert wav_2.exists(), wav_2

print("Local data check passed.")

VoxCeleb1 dev speakers: 1211
VoxCeleb1 test speakers: 40
iden_split lines: 153516
veri_test2 lines: 37611
Local data check passed.


**2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных**

Построение современных дикторских моделей, как правило, выполняется с использованием нейросетевых архитектур. В последние годы наилучшие результаты демонстрируют [трансформерные модели](https://arxiv.org/pdf/1706.03762), существенно превосходящие по качеству более ранние свёрточные нейросетевые архитектуры. В рамках настоящего пункта предлагается выполнить адаптацию нейросетевой архитектуры [Whisper tiny](https://arxiv.org/pdf/2212.04356) под решение задачи генерации дикторских моделей (дикторских эмбеддингов). *Дикторский эмбеддинг* – это высокоуровневый вектор-признаков, состоящий, например, из 128, 256 и т.п. значений, содержащий особенности голоса конкретного человека. При решении задачи распознавания диктора можно выделить эталонные и тестовые дикторские эмбеддинги. *Эталонные эмбеддинги* формируются на этапе регистрации дикторской модели определённого человека и находятся в некотором хранилище данных. *Тестовые эмбеддинги* формируются на этапе непосредственного использования системы голосовой биометрии на практике, когда некоторый пользователь пытается получить доступ к соответствующим ресурсам. Система голосовой биометрии сравнивает по определённой метрике эталонные и тестовые эмбеддинги, формируя оценку сравнения, которая, после её обработки блоком принятия решения, позволяет сделать вывод о том, эмбеддинги одинаковых или разных дикторов сравниваются между собой.

Необходимо отметить, что построение дикторских моделей, как правило, требует наличия *акустических признаков*, вычисленных для звукозаписей тренировочной, валидационной и тестовой баз данных. В качестве примера подобных признаков в рамках настоящей лабораторной работы воспользуемся *логарифмами энергий на выходе мел-банка фильтров*. Важно отметить, что оригинальная модель Whisper tiny обучалась на 80-мерных признаках, причём входной сигнал перед обработкой путём дополнения с краю приводится к фиксированной длительности равной 30 секундам. В рамках настоящей лабораторной работы дополнение сигнала до 30 секунд не выполняется. Дополнительно, перед вычислением акустических признаков, осуществляется применение [нормализации экземпляра](https://docs.pytorch.org/docs/stable/generated/torch.nn.InstanceNorm1d.html) (instance normalization), которая выравнивает амплитуды всех сигналов в тренировочной выборке, стабилизируя процедуру сходимости нейросетевой модели, выполняющей построение дикторских моделей, на этапе обучения. Подобная нормализация отсутствует в процедуре обучения оригинальных моделей семейства Whisper.

После того, как акустические признаки подготовлены, они могут быть переданы на блок построения дикторских моделей. Как правило, устройство современных дикторских моделей соответствует структуре [x-векторных архитектур](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf). Эти архитектуры состоят из четырёх ключевых элементов: 

1. **Фреймовый уровень.** Предназначен для формирования локальных представлений голоса конкретного человека. На этом уровне как раз и могут быть использованы нейросетевые архитектуры на базе трансормерных моделей. При организации фреймового уровня в рамках настоящей лабораторной работы предлагается использовать предобученный кодировщик трансформерной нейросетевой архитектуры Whisper tiny, а также два [нейросетевых блока с временной задержкой](https://www.cs.toronto.edu/~hinton/absps/waibelTDNN.pdf) (TDNN, time-delay neural betwork), которые выполняют согласование кодировщика Whisper tiny с оставшейся частью нейронной сети. Использование мощных, предварительно обученных на большом объёме размеченных или неразмеченных данных трансформерных моделей, таких как [wav2vec 2.0](https://arxiv.org/pdf/2006.11477), [HuBERT](https://arxiv.org/pdf/2106.07447), [Whisper](https://arxiv.org/pdf/2212.04356) и т.п., значительно упрощает решение множества разнообразных *целевых задач* (downstream tasks), связанных с обработкой аудиосигналов, ускоряя процедуру сходимости на этапе обучения и улучшая качество работы моделей в процессе тестирования относительно моделей, обученных из случайной начальной инициализации.

2. **Уровень статистического пулинга** позволяет сформировать промежуточный вектор-признаков, фиксированной длины, которая является одинаковой для звукозаписи любой длительности. В ходе работы блока статистического пулинга происходит удаление временной размерности, присутствующей в картах-признаков. Это достигается путём выполнения процедуры усреднения карт-признаков вдоль оси времени. Выходом уровня статистического пулинга являются вектор среднего и вектор среднеквадратического отклонения, вычисленные на основе карт-признаков. Эти вектора конкатенируются и передаются для дальнейшей обработки на сегментом уровне.

3. **Сегментный уровень.** Предназначен для трансформации промежуточного вектора, как правило, высокой размерности, в компактный вектор-признаков, представляющий собой дикторский эмбеддинг. Необходимо отметить, что на сегментном уровне расположены один или несколько полносвязных нейросетевых слоёв, а обработка данных выполняется по отношению ко всей звукозаписи, а не только к некоторому её локальному контексту, как на фреймовом уровне.

4. **Уровень выходного слоя.** Представляет полносвязный слой с softmax-функциями активации. Количество активаций равно числу дикторов в тренирочной выборке. На вход выходноя слоя подаётся дикторский эмбеддинг, а на выходе – формируется набор апостериорных вероятностей, определяющих принадлежность эмбеддинга к одному из дикторских классов в тренировочной выборке. Необходимо отметить, что, как правило, в современных нейросетевых системах построения дикторских моделей выходной слой используется только на этапе обучения параметров и на этапе тестирования не применяется. На этапе тестирования используются только три первых уровня архитектуры.

Обучение модели генерации дикторских эмбеддингов выполняется путём решения задачи *классификации* или, выражаясь терминами из области биометрии, *идентификации на закрытом множестве* (количество дикторских меток является строго фиксированным). В качестве используемой стоимостной функции выступает *категориальная кросс-энтропия*. Обучение выполняется с помощью мини-батчей, содержащих короткие фрагменты карт акустических признаков (длительностью несколько секунд) различных дикторов из тренировочной базы данных. Обучение на коротких фрагментах позволяет избежать сильного переобучения нейросетевой модели. При выполнении процедуры обучения требуется подобрать набор гиперпараметров, выбрать обучения и метод численной оптимизации. Обучение нейросетевых моделей при наличие претрейнов добавляет определённую специфику в процедуру обучения: использование более низкого значения параметра сходимости в случае, если вся нейросетевая модель *«разморожена»* (обучаются все веса модели на этапе тренировки), *«заморозка»* предобученной части модели (обучается часть весов модели на этапе тренировки) для ускорения процедуры сходимости и быстрой проверки гипотез и т.п.

Для успешного выполнения настоящего пункта необходимо сделать следующее:

1. Сгенерировать списки тренировочных и валидационных данных на основе идентификационного протокола базы VoxCeleb1, содержащегося в файле **../data/voxceleb1_test/iden_split.txt**. При генерации списков требуется исключить из них звукозаписи дикторов, которые входят в базу [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip). Это позволит выполнить тестирования обученных блоков генерации дикторских моделей на протоколе [VoxCeleb1-O cleaned](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt), который составлен по отношению к данным из VoxCeleb1 test set и позволяет проверить качество работы биометрической системы в рамках открытой задачи верификации диктора.

2. Инициализировать обучаемый экстрактор дикторских эмбеддингов и вгрузить в его фреймовый уровень веса предобученного кодировщика модели Whisper tiny. Необходимо отметить, что Whisper tiny не является единственной моделью семейства [Whisper](https://github.com/openai/whisper/tree/main). Так же в общедоступных источниках могут быть найдены веса предобученных моделей Whisper base/small/medium/large. Однако, вычислительная стоимость этих моделей в сравнении с Whisper tiny является выше при более высокой точности. Для достижения более высокого качества экстрактора дикторских эмбеддингов предлагается разморозить слои кодировщика Whisper tiny на этапе обучения. 

3. Инициализировать загрузчики тренировочной и валидационной выборок, используемые в рамках решения задачи идентификации на закрытом множестве, а также тестовой выборки, для оценки ошибки системы распознавания в рамках решения задачи верификации.

4. Инициализировать оптимизатор и планировщик для выполнения процедуры обучения.

5. Описать процедуру валидации блока построения дикторских моделей.

6. Описать процедуру обучения и запустить её, контролируя значения стоимостной функции и доли правильных ответов на тренировочном и валидационном множествах, а также величину равновероятной ошибки при решении задачи верификации на тестовом множестве.

In [6]:
# Select hyperparameters

nOut              = 512                                  # embedding size

# Loss function for angular losses
margin            = 0.35                                 # margin parameter
scale             = 32.0                                 # scale parameter

# Train dataloader (close-set identification)
max_frames_train  = 400                                  # number of frame to train
#train_path        = '../data/voxceleb1_dev/wav'         # path to train wav files
train_path        = train_path                           # path to train wav files
batch_size_train  = 128                                  # batch size to train
pin_memory        = False                                # pin memory
num_workers_train = 4                                    # number of workers to train
shuffle           = True                                 # shuffling of training examples

# Validation dataloader (close-set identification)
max_frames_val    = 400                                  # number of frame to validate
#val_path          = '../data/voxceleb1_dev/wav'         # path to val wav files
val_path          = val_path                             # path to val wav files
batch_size_val    = 128                                  # batch size to validate
num_workers_val   = 4                                    # number of workers to validate

# Test dataloader (open-set verification)
max_frames_test   = 400                                  # number of frame to test
#test_path         = '../data/voxceleb1_test/wav/'       # path to val wav files
test_path         = test_path                            # path to val wav files
batch_size_test   = 1                                    # batch size to test
num_workers_test  = 4                                    # number of workers to test
num_eval          = 5                                    # number of enroll/test chunks for scoring

# Optimizer
lr                = 0.0001                               # learning rate value # 0.0001
weight_decay      = 0                                    # weight decay value

# Scheduler
val_interval      = 5                                    # frequency of validation step
max_epoch         = 40                                   # number of epoches

# Augmentation
musan_path        = musan_path                           # path to splitted SLR17 dataset
rir_path          = rir_path                             # path to SLR28 dataset

In [7]:
# Generate train/validation data lists for close-set speaker identification
train_list = []
val_list   = []

with open(iden_split_path, 'r') as f:
    lines = f.readlines()
    
black_list = {
    name for name in os.listdir(test_path)
    if name.startswith("id") and os.path.isdir(os.path.join(test_path, name))}  # exclude speaker IDs from VoxCeleb1 test set
num_train_spk = []                                                              # number of train speakers

for line in lines:
    line   = line.strip().split(' ')
    spk_id = line[1].split('/')[0]
    
    if not (spk_id in black_list):
        num_train_spk.append(spk_id)
        
    else:
        continue
    
    # Train list
    if (line[0] == '1'):
        train_list.append(' '.join([spk_id, line[1]]))
    
    # Validation list
    elif (line[0] == '2'):
        val_list.append(' '.join([spk_id, line[1]]))
        
num_train_spk = len(set(num_train_spk))

In [8]:
# Initialize train dataloader (with augmentation)
train_dataset = train_dataset_loader(train_list=train_list, 
                                     max_frames=max_frames_train, 
                                     train_path=train_path, 
                                     augment=False, 
                                     musan_path=musan_path, 
                                     rir_path=rir_path)

train_loader  = DataLoader(train_dataset, 
                           batch_size=batch_size_train, 
                           pin_memory=pin_memory, 
                           num_workers=num_workers_train, 
                           shuffle=shuffle)

# Initialize validation dataloader
val_dataset = test_dataset_loader(test_list=val_list, max_frames=max_frames_val, test_path=val_path)
val_loader  = DataLoader(val_dataset, batch_size=batch_size_val, num_workers=num_workers_val)

print("augment:", train_dataset.augment)
print("train size:", len(train_dataset))

augment: False
train size: 134000


In [9]:
# Load test protocol
with open(VOX1_TEST_ROOT/'veri_test2.txt', 'r') as f:
    verif_protocol_lines = f.readlines()
    
# Get a list of unique file names
files = list(itertools.chain(*[x.strip().split()[-2:] for x in verif_protocol_lines]))
setfiles = list(set(files))
setfiles.sort()

# Initialize test dataloader
test_dataset = test_dataset_loader_verification(setfiles, test_path=test_path, eval_frames=max_frames_test, num_eval=num_eval)
test_loader  = DataLoader(test_dataset, 
                          batch_size=batch_size_test, 
                          shuffle=False, 
                          num_workers=num_workers_test, 
                          drop_last=False, 
                          sampler=None)

In [10]:
# Initialize model
preprocessor        = ExtractWhisperFbanks80(pad_data=False)
feat_extractor      = WhisperEncoder.build_model(model_name="tiny", load_weight=True, n_layer=-1) # Whisper tiny download to 
                                                                                                  # ~/.cache/whisper/tiny.pt
frame_level_block   = TDNNSimple(n_layers=1, in_ch=384, out_ch=1024)
pooling_block       = StatPoolLayer.by_string_mode(mode="MV")
segment_level_block = MaxoutSegmentLevel(input_dim=2048, output_dim=512, enable_batch_norm=True)

whisper_sr_model    = SSLDownstream(preproc=preprocessor,
                                    feat_extractor=feat_extractor,
                                    frame_level_block=frame_level_block,
                                    pooling_block=pooling_block,
                                    segment_level_block=segment_level_block,
                                    freeze_feats=False)
trainfunc           = AAMSoftmaxLoss(nOut=nOut, nClasses=num_train_spk, margin=margin, scale=scale)
main_model          = MainModel(whisper_sr_model, trainfunc).cuda()

Initialised AAM softmax margin 0.350 scale 32.000.


In [11]:
# Initialize optimizer and scheduler
optimizer = SGDOptimizer(main_model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = OneCycleLRScheduler(optimizer, 
                                pct_start=0.05, 
                                cycle_momentum=False, 
                                max_lr=lr, 
                                div_factor=10, 
                                final_div_factor=10000, 
                                total_steps=max_epoch*len(train_loader))

Initialised SGD optimizer.
Initialised OneCycle LR scheduler.


In [ ]:
start_epoch = 0
checkpoint_flag = False

if checkpoint_flag:
    start_epoch = loadParameters(main_model, optimizer, scheduler, path='../data/lab6_models/lab6_model_0004.pth')
    start_epoch = start_epoch + 1

# Train model
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    if (num_epoch + 1)%val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)                               # compute accuracy for validation set
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        feats = extract_features(main_model, test_loader)                                # compute enroll/test speaker embeddings
        all_scores, all_labels, all_trials = compute_scores(feats, verif_protocol_lines) # compute scores between enroll and test speaker models
        tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)                   # devide the scores into target and impostor scores
        EER, _ = get_eer(tar_scores, imp_scores)                                         # compute EER, %
        
        print("Epoch {:1.0f}, EER (test set) {:2.3f}%".format(num_epoch, EER))
        
        saveParameters(main_model, optimizer, scheduler, num_epoch, path='../data/lab6_models')

**3. Дополнительные задания по обучению дикторской модели на основе трансформерной архитектуры**

В рамках настоящего пункта предлагается выбрать и решить одно из дополнительных заданий на выбор:

1. Переведя параметр load_weight кодировщика Whisper tiny в режим False, попробуйте обучить трансформерную нейронную сеть без предварительно загруженных в неё весов кодировщика. Сравните полученное качество модели с тем, что было в пункте 2. При проведении исследования разрешается менять любые настройки процедуры обучения, в частности, параметр сходимости.

2. Переведя параметр freeze_feats кодировщика Whisper tiny в режим True, попробуйте обучить трансформерную нейронную сеть с «замороженным» кодировщиком. Сравните полученное качество модели с тем, что было в пункте 2. При проведении исследования разрешается менять любые настройки процедуры обучения, в частности, параметр сходимости.

3. В настройках процедуры обучения рассогласуйте параметр, отвечающий за максимальное число фреймов, используемых на этапе обучения, валидации и тестирования, например, выставив его равным 400 на этапе обучения и 1000 на этапах валидации и тестирования. Попробуйте обучить трансформерную нейронную сеть для выбранных настроек. Сравните полученное качество модели с тем, что было в пункте 2.

4. Попробуйте заменить кодировщика Whisper tiny на Whisper модель больших размеров base/small/medium/large и повторите процедуру обучения из пункта 2. Сравните полученное качество модели с тем, что было в пункте 2.

In [15]:
# Initialize model
preprocessor        = ExtractWhisperFbanks80(pad_data=False)
feat_extractor      = WhisperEncoder.build_model(model_name="tiny", load_weight=True, n_layer=-1) # Whisper tiny download to 
                                                                                                  # ~/.cache/whisper/tiny.pt
frame_level_block   = TDNNSimple(n_layers=1, in_ch=384, out_ch=1024)
pooling_block       = StatPoolLayer.by_string_mode(mode="MV")
segment_level_block = MaxoutSegmentLevel(input_dim=2048, output_dim=512, enable_batch_norm=True)

whisper_sr_model    = SSLDownstream(preproc=preprocessor,
                                    feat_extractor=feat_extractor,
                                    frame_level_block=frame_level_block,
                                    pooling_block=pooling_block,
                                    segment_level_block=segment_level_block,
                                    freeze_feats=True)
trainfunc           = AAMSoftmaxLoss(nOut=nOut, nClasses=num_train_spk, margin=margin, scale=scale)
main_model          = MainModel(whisper_sr_model, trainfunc).cuda()

Initialised AAM softmax margin 0.350 scale 32.000.


In [16]:
# Initialize optimizer and scheduler
optimizer = SGDOptimizer(main_model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = OneCycleLRScheduler(optimizer, 
                                pct_start=0.05, 
                                cycle_momentum=False, 
                                max_lr=lr, 
                                div_factor=10, 
                                final_div_factor=10000, 
                                total_steps=max_epoch*len(train_loader))

Initialised SGD optimizer.
Initialised OneCycle LR scheduler.


In [ ]:
start_epoch = 0
checkpoint_flag = True

if checkpoint_flag:
    start_epoch = loadParameters(
        main_model,
        optimizer,
        scheduler,
        path="../data/lab6_models_frozen/lab6_model_0015.pth",
    )
    start_epoch = start_epoch + 1

print("Continue from epoch:", start_epoch)

Continue from epoch: 16


In [ ]:
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    saveParameters(main_model, optimizer, scheduler, num_epoch, path="../data/lab6_models_frozen")

    if (num_epoch + 1) % val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        feats = extract_features(main_model, test_loader)
        all_scores, all_labels, all_trials = compute_scores(feats, verif_protocol_lines)
        tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)
        EER, _ = get_eer(tar_scores, imp_scores)
        
        print("Epoch {:1.0f}, EER (test set) {:2.3f}%".format(num_epoch, EER))

Epoch 16, Batch 1, LR 0.000070 Loss 13.286860, Accuracy 43.750%
Epoch 16, Batch 2, LR 0.000070 Loss 13.137475, Accuracy 44.531%
Epoch 16, Batch 3, LR 0.000070 Loss 12.925426, Accuracy 47.135%
Epoch 16, Batch 4, LR 0.000070 Loss 13.026338, Accuracy 45.117%
Epoch 16, Batch 5, LR 0.000070 Loss 12.973232, Accuracy 46.719%
Epoch 16, Batch 6, LR 0.000070 Loss 13.027405, Accuracy 47.396%
Epoch 16, Batch 7, LR 0.000070 Loss 12.963701, Accuracy 49.107%
Epoch 16, Batch 8, LR 0.000070 Loss 13.003489, Accuracy 48.535%
Epoch 16, Batch 9, LR 0.000070 Loss 13.045950, Accuracy 47.743%
Epoch 16, Batch 10, LR 0.000070 Loss 13.086031, Accuracy 47.344%
Epoch 16, Batch 11, LR 0.000070 Loss 13.077204, Accuracy 47.088%
Epoch 16, Batch 12, LR 0.000070 Loss 13.067645, Accuracy 47.331%
Epoch 16, Batch 13, LR 0.000070 Loss 13.074458, Accuracy 47.356%
Epoch 16, Batch 14, LR 0.000070 Loss 13.090601, Accuracy 46.819%
Epoch 16, Batch 15, LR 0.000070 Loss 13.101308, Accuracy 46.979%
Epoch 16, Batch 16, LR 0.000070 Lo

In [10]:
start_epoch = 0
checkpoint_flag = True

if checkpoint_flag:
    start_epoch = loadParameters(
        main_model,
        optimizer,
        scheduler,
        path="../data/lab6_models_frozen/lab6_model_0017.pth",
    )
    start_epoch = start_epoch + 1

print("Continue from epoch:", start_epoch)

Continue from epoch: 18


In [ ]:
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    saveParameters(main_model, optimizer, scheduler, num_epoch, path="../data/lab6_models_frozen")

    if (num_epoch + 1) % val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        feats = extract_features(main_model, test_loader)
        all_scores, all_labels, all_trials = compute_scores(feats, verif_protocol_lines)
        tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)
        EER, _ = get_eer(tar_scores, imp_scores)
        
        print("Epoch {:1.0f}, EER (test set) {:2.3f}%".format(num_epoch, EER))

Epoch 18, Batch 1, LR 0.000062 Loss 12.903332, Accuracy 46.094%
Epoch 18, Batch 2, LR 0.000062 Loss 12.715977, Accuracy 48.828%
Epoch 18, Batch 3, LR 0.000062 Loss 12.730205, Accuracy 47.917%
Epoch 18, Batch 4, LR 0.000062 Loss 12.768945, Accuracy 48.242%
Epoch 18, Batch 5, LR 0.000062 Loss 12.802649, Accuracy 48.438%
Epoch 18, Batch 6, LR 0.000062 Loss 12.844752, Accuracy 48.047%
Epoch 18, Batch 7, LR 0.000062 Loss 12.884362, Accuracy 47.545%
Epoch 18, Batch 8, LR 0.000062 Loss 12.905761, Accuracy 47.754%
Epoch 18, Batch 9, LR 0.000062 Loss 12.875161, Accuracy 48.264%
Epoch 18, Batch 10, LR 0.000062 Loss 12.854472, Accuracy 48.672%
Epoch 18, Batch 11, LR 0.000062 Loss 12.819445, Accuracy 49.148%
Epoch 18, Batch 12, LR 0.000062 Loss 12.787760, Accuracy 50.065%
Epoch 18, Batch 13, LR 0.000062 Loss 12.766137, Accuracy 50.481%
Epoch 18, Batch 14, LR 0.000062 Loss 12.798260, Accuracy 49.721%
Epoch 18, Batch 15, LR 0.000062 Loss 12.744402, Accuracy 50.677%
Epoch 18, Batch 16, LR 0.000062 Lo

In [10]:
start_epoch = 0
checkpoint_flag = True

if checkpoint_flag:
    start_epoch = loadParameters(
        main_model,
        optimizer,
        scheduler,
        path="../data/lab6_models_frozen/lab6_model_0021.pth",
    )
    start_epoch = start_epoch + 1

print("Continue from epoch:", start_epoch)

Continue from epoch: 22


In [ ]:
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    saveParameters(main_model, optimizer, scheduler, num_epoch, path="../data/lab6_models_frozen")

    if (num_epoch + 1) % val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        feats = extract_features(main_model, test_loader)
        all_scores, all_labels, all_trials = compute_scores(feats, verif_protocol_lines)
        tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)
        EER, _ = get_eer(tar_scores, imp_scores)
        
        print("Epoch {:1.0f}, EER (test set) {:2.3f}%".format(num_epoch, EER))

Epoch 22, Batch 1, LR 0.000046 Loss 12.525718, Accuracy 51.562%
Epoch 22, Batch 2, LR 0.000046 Loss 12.514474, Accuracy 54.688%
Epoch 22, Batch 3, LR 0.000046 Loss 12.620824, Accuracy 53.906%
Epoch 22, Batch 4, LR 0.000046 Loss 12.651795, Accuracy 50.781%
Epoch 22, Batch 5, LR 0.000046 Loss 12.706976, Accuracy 50.312%
Epoch 22, Batch 6, LR 0.000046 Loss 12.704395, Accuracy 50.521%
Epoch 22, Batch 7, LR 0.000046 Loss 12.687835, Accuracy 50.335%
Epoch 22, Batch 8, LR 0.000046 Loss 12.615289, Accuracy 50.977%
Epoch 22, Batch 9, LR 0.000046 Loss 12.622534, Accuracy 51.128%
Epoch 22, Batch 10, LR 0.000046 Loss 12.609599, Accuracy 51.172%
Epoch 22, Batch 11, LR 0.000046 Loss 12.552278, Accuracy 51.705%
Epoch 22, Batch 12, LR 0.000046 Loss 12.524768, Accuracy 51.953%
Epoch 22, Batch 13, LR 0.000046 Loss 12.533261, Accuracy 51.683%
Epoch 22, Batch 14, LR 0.000046 Loss 12.552796, Accuracy 51.562%
Epoch 22, Batch 15, LR 0.000046 Loss 12.565974, Accuracy 51.562%
Epoch 22, Batch 16, LR 0.000046 Lo

In [10]:
start_epoch = 0
checkpoint_flag = True

if checkpoint_flag:
    start_epoch = loadParameters(
        main_model,
        optimizer,
        scheduler,
        path="../data/lab6_models_frozen/lab6_model_0025.pth",
    )
    start_epoch = start_epoch + 1

print("Continue from epoch:", start_epoch)

Continue from epoch: 26


In [ ]:
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    saveParameters(main_model, optimizer, scheduler, num_epoch, path="../data/lab6_models_frozen")

    if (num_epoch + 1) % val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        feats = extract_features(main_model, test_loader)
        all_scores, all_labels, all_trials = compute_scores(feats, verif_protocol_lines)
        tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)
        EER, _ = get_eer(tar_scores, imp_scores)
        
        print("Epoch {:1.0f}, EER (test set) {:2.3f}%".format(num_epoch, EER))

Epoch 26, Batch 1, LR 0.000030 Loss 12.350817, Accuracy 52.344%
Epoch 26, Batch 2, LR 0.000030 Loss 12.061019, Accuracy 56.641%
Epoch 26, Batch 3, LR 0.000030 Loss 11.989613, Accuracy 58.594%
Epoch 26, Batch 4, LR 0.000030 Loss 12.087289, Accuracy 57.422%
Epoch 26, Batch 5, LR 0.000030 Loss 12.136142, Accuracy 55.938%
Epoch 26, Batch 6, LR 0.000030 Loss 12.149972, Accuracy 55.339%
Epoch 26, Batch 7, LR 0.000030 Loss 12.162890, Accuracy 55.469%
Epoch 26, Batch 8, LR 0.000030 Loss 12.155986, Accuracy 55.957%
Epoch 26, Batch 9, LR 0.000030 Loss 12.221827, Accuracy 54.861%
Epoch 26, Batch 10, LR 0.000030 Loss 12.170758, Accuracy 55.156%
Epoch 26, Batch 11, LR 0.000030 Loss 12.213152, Accuracy 54.972%
Epoch 26, Batch 12, LR 0.000030 Loss 12.224473, Accuracy 54.883%
Epoch 26, Batch 13, LR 0.000030 Loss 12.182706, Accuracy 55.589%
Epoch 26, Batch 14, LR 0.000030 Loss 12.187599, Accuracy 55.692%
Epoch 26, Batch 15, LR 0.000030 Loss 12.236418, Accuracy 55.000%
Epoch 26, Batch 16, LR 0.000030 Lo

In [17]:
start_epoch = 0
checkpoint_flag = True

if checkpoint_flag:
    start_epoch = loadParameters(
        main_model,
        optimizer,
        scheduler,
        path="../data/lab6_models_frozen/lab6_model_0035.pth",
    )
    start_epoch = start_epoch + 1

print("Continue from epoch:", start_epoch)

Continue from epoch: 36


In [18]:
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    saveParameters(main_model, optimizer, scheduler, num_epoch, path="../data/lab6_models_frozen")

    if (num_epoch + 1) % val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        feats = extract_features(main_model, test_loader)
        all_scores, all_labels, all_trials = compute_scores(feats, verif_protocol_lines)
        tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)
        EER, _ = get_eer(tar_scores, imp_scores)
        
        print("Epoch {:1.0f}, EER (test set) {:2.3f}%".format(num_epoch, EER))

Epoch 36, Batch 1, LR 0.000003 Loss 11.747827, Accuracy 57.031%
Epoch 36, Batch 2, LR 0.000003 Loss 11.548123, Accuracy 60.547%
Epoch 36, Batch 3, LR 0.000003 Loss 11.688329, Accuracy 59.375%
Epoch 36, Batch 4, LR 0.000003 Loss 11.837874, Accuracy 58.008%
Epoch 36, Batch 5, LR 0.000003 Loss 11.828526, Accuracy 57.969%
Epoch 36, Batch 6, LR 0.000003 Loss 11.819954, Accuracy 58.203%
Epoch 36, Batch 7, LR 0.000003 Loss 11.834738, Accuracy 58.817%
Epoch 36, Batch 8, LR 0.000003 Loss 11.852186, Accuracy 58.887%
Epoch 36, Batch 9, LR 0.000003 Loss 11.923457, Accuracy 58.247%
Epoch 36, Batch 10, LR 0.000003 Loss 11.953553, Accuracy 57.969%
Epoch 36, Batch 11, LR 0.000003 Loss 11.909628, Accuracy 58.239%
Epoch 36, Batch 12, LR 0.000003 Loss 11.898325, Accuracy 58.073%
Epoch 36, Batch 13, LR 0.000003 Loss 11.912137, Accuracy 57.873%
Epoch 36, Batch 14, LR 0.000003 Loss 11.926394, Accuracy 57.924%
Epoch 36, Batch 15, LR 0.000003 Loss 11.945554, Accuracy 57.969%
Epoch 36, Batch 16, LR 0.000003 Lo

### Почему уменьшилась точность?
- Whisper tiny изначально обучался для распознавания **речи**, а не для распознавания **дикторов**. При дообучении с freeze_feats=False, веса Whisper подстраиваются под задачу, а при freeze_feats=False адаптация происходит только на слоях TDNN и пуллинга.

In [ ]:
preprocessor = ExtractWhisperFbanks80(pad_data=False)
feat_extractor = WhisperEncoder.build_model(model_name="tiny", load_weight=True, n_layer=-1)
frame_level_block = TDNNSimple(n_layers=1, in_ch=384, out_ch=1024)
pooling_block = StatPoolLayer.by_string_mode(mode="MV")
segment_level_block = MaxoutSegmentLevel(input_dim=2048, output_dim=512, enable_batch_norm=True)
whisper_sr_model = SSLDownstream(preproc=preprocessor, feat_extractor=feat_extractor,
                                 frame_level_block=frame_level_block, pooling_block=pooling_block,
                                 segment_level_block=segment_level_block, freeze_feats=False)
trainfunc = AAMSoftmaxLoss(nOut=nOut, nClasses=num_train_spk, margin=margin, scale=scale)
model = MainModel(whisper_sr_model, trainfunc).cuda()

checkpoint = torch.load("../data/lab6_models/lab6_model_0039.pth")
model.load_state_dict(checkpoint['model'])
print(f"Загружена эпоха {checkpoint.get('num_epoch', '?')}")

# Loss и Accuracy
train_loss, train_acc = test_network(train_loader, model)  # test_network не обучает
print(f"Epoch 39, Loss (train set) {train_loss:.6f}, Accuracy (train set) {train_acc:.3f}%")

# Accuracy на валидационном множестве
_, val_acc = test_network(val_loader, model)
print(f"Epoch 39, Accuracy (validation set) {val_acc:.3f}%")

# EER на тестовом множестве
feats = extract_features(model, test_loader)
all_scores, all_labels, _ = compute_scores(feats, verif_protocol_lines)
tar_scores, imp_scores = tar_imp_hists(all_scores, all_labels)
eer, _ = get_eer(tar_scores, imp_scores)
print(f"Epoch 39, EER (test set) {eer:.3f}%")

Initialised AAM softmax margin 0.350 scale 32.000.
Загружена эпоха 39
Epoch 39, Loss (train set) 1.879172, Accuracy (train set) 97.755%
Epoch 39, Accuracy (validation set) 77.220%
Epoch 39, EER (test set) 6.524%


**4. Контрольные вопросы**

1. Что такое трансформерные нейросетевые модели и для какой задачи они были изначально предложены?  
*На основе механизмов внимания для машинного перевода*
2. Назовите основные отличия трансформерных нейросетевых архитектур от свёрточных?  
*Обрабатывают последовательность целиком(глобальные связи), а не строят иерархию признаков*

3. Как устроена архитектура Whisper и для решения каких задач она была изначально разработана?  
*Wihsper - это кодер-декодер трансформер. Был разработан для автоматического распознавания речи (ASR) и перевода на английский*

4. Из каких основных нейросетевых слоёв состоит архитектура Whisper?
*Основные слои Whisper: Conv1d (слой свертки для спектрограммы), позиционное кодирование (порядок фреймов), блоки внимания, нормализация.*

5. Какие разновидности архитектуры Whisper существуют и чем они отличаются друг от друга?  
*Существуют: tiny, base, small, medium, largе. Отличаются вычислительной стоимостью: количеством параметров, слоёв, голов внимания и размерностью эмбеддингов*

6. Что такое предобученная нейросетевая модель и для каких целей она используется?  
*Предобученная модель - нейросеть, обученная для какой-либо задачи. Используется для дообучения под конкретную задачу (ускорение сходимости, улучшение качества, работа с малыми данными)*

7. В чём состоит основное отличие в политике изменения параметра сходимости на этапе обучения при наличии преобученной модели от ситуации, когда она отсутствует?  
*При наличии предобученной модели learning rate меньше (только адаптация под задачу) или замораживаются первые слои. При обучении с нуля learning rate выше.*

8. Назовите примеры трансформерных нейросетевых претрейнов для обработки аудиосигналов и кратко опишите, как они были получены?  
*и **wav2vec 2.0** (контрастное обучение на неразмеченной речи), **HuBERT** (скрытые единицы через кластеризацию), **Whisper** (слабое обучение на 680k часов аудио с транскрипцией).*

9. Как реализовать блок построения дикторских моделей на основе архитектуры Whisper?  
*взять замороженный или дообучаемый кодировщик Whisper, добавить TDNN, слой статистического пулинга, сегментный уровень, получить эмбеддинг и обучить на задаче идентификации дикторов.*

**5. Список литературы**

1. Bai Z., Zhang X.-L., Chen J. Speaker recognition based on deep learning: an overview // 	arXiv:2012.00931 [eess.AS] ([ссылка](https://arxiv.org/pdf/2012.00931.pdf)).

2. Hansen J.H.L., Hasan T. Speaker recognition by machines and humans: a tutorial review // IEEE Signal Processing Magazine, 2015. V. 32. № 6. P. 74–99 ([ссылка](https://www.researchgate.net/publication/282940395_Speaker_Recognition_by_Machines_and_Humans_A_tutorial_review)).

3. Vaswani A. et al. Attention is all you need // arXiv:1706.03762 [cs.CL] ([ссылка](https://arxiv.org/pdf/1706.03762)).

4. Radford A. et al. Robust speech recognition via large-scale weak supervision // arXiv:2212.04356 [eess.AS] ([ссылка](https://arxiv.org/pdf/2212.04356)).